# Kelvin DtN-spectrum archive

This notebook promotes the `examples/kelvin_transformation/DtN_spectrum` Python corpus into a docs-layer archive before pruning the standalone research scripts. It stores full source text and SHA-256 hashes in `kelvin_dtn_spectrum_archive_results.json`.

The productionized parts now live in `src/radia/open_boundary` and `validation_test/open_boundary`; historical act scripts are retained here as source records rather than as runnable files under `examples/`.


## Theory Route

Every open-boundary closure in this corpus is treated as an approximation of the same exterior Dirichlet-to-Neumann operator `Lambda_ext` on the truncation surface. For a sphere, the exact static ladder is `lambda_n = -(n+1)/R` in 3D and `lambda_n = -n/R` in 2D. The key lesson is that a coarse Kelvin exterior can already reproduce the low multipoles that dominate compact sources; the remaining error is read from the DtN spectrum rather than guessed from exterior element count.

The practical mesh-control message is: spend resolution on the interface geometry and the source-side singularities, not on the transformed exterior volume. In the archived scripts this appears as the `p >= n` threshold, the curved-boundary floor `~(h/R)^(2k)`, the optimal-radius estimate near `R/a ~ 3`, corner `hp` behavior, and the measured DOF-cost gap for exterior volume refinement.

The productionized pieces now live in `src/radia/open_boundary` and `validation_test/open_boundary`; this notebook keeps the historical script source and the Markdown reading guide together so the examples tree can stay small without losing the derivation trail.


## Reading Guide

| Act | Role | Main lesson |
|---|---|---|
| 0 | The coarse-mesh question | Low multipoles are accurate on coarse Kelvin meshes; p/geometry order matters more than exterior volume density. |
| 1 | One operator, one spectrum | Kelvin, BEM, Robin, PML, and finite truncation are compared as partial views of the same DtN spectrum. |
| 2 | Mesh-adequacy criterion | The closed-form adequacy rule combines source multipole content, polynomial image order, and curved-boundary geometry floor. |
| 3 | Kelvin realizes the operator | Kelvin compactification gives a sparse FEM realization of the exterior operator, including vector/A-form caveats. |
| 4 | Sparse BEM viewpoint | The Kelvin exterior Schur complement is the sparse FEM counterpart of a dense boundary operator. |
| 5 | Beyond vacuum and the sphere | Material-aware and non-spherical exterior operators motivate the FEM-built DtN route. |
| 6 | Time axis | Eddy/diffusion DtN becomes a passive Cauer/CLN ladder in `sqrt(s)`; these results are now covered by open-boundary tests. |
| 7 | Radiation and infinite elements | High-frequency/PML/IE comparisons are archived here; the shipped Radia scope remains MQS/Laplace. |
| 8 | Application bridge | The material-aware DtN becomes a stream-function coil-design kernel when iron breaks free-space Biot-Savart. |

The capstone historical script is `act7_22_dtn_spectrum_consolidated.py`: it puts every closure and regime on the same per-mode DtN-defect yardstick. Its full source is stored in `kelvin_dtn_spectrum_archive_results.json`; the readable datasheet is `docs/open_boundary/DTN_SPECTRUM_COMPARISON.md`.


## Archive Policy

This notebook intentionally does not rerun all 122 historical scripts. Many are derivation probes, cost studies, or long-running FEM/BEM comparisons. Instead, the synchronized JSON stores full source text, line count, byte count, and SHA-256 for each script. The notebook output records the archive inventory and the small set of scripts that had validation/test references at migration time.

For future debugging, recover exact historical source from `kelvin_dtn_spectrum_archive_results.json`; for maintained behavior, use `src/radia/open_boundary`, `src/radia/infinite_element.py`, and `validation_test/open_boundary`.


In [1]:
from pathlib import Path
import json
import re
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'kelvin_examples_migration.py').exists():
    NOTEBOOK_DIR = Path('docs/kelvin').resolve()
sys.path.insert(0, str(NOTEBOOK_DIR))

from kelvin_examples_migration import build_migration_report, build_source_archive, markdown_table, write_report_json

archive_path = NOTEBOOK_DIR / 'kelvin_dtn_spectrum_archive_results.json'
report = build_migration_report()
fresh_archive = build_source_archive(
    report,
    lanes=['open_boundary_src_api_or_validation', 'validation_test_or_src_api_locked', 'validation_test_candidate'],
    path_contains='examples/kelvin_transformation/DtN_spectrum/',
    include_source=True,
)
archive = fresh_archive
if archive_path.exists():
    existing = json.loads(archive_path.read_text(encoding='utf-8'))
    existing_count = existing.get('summary', {}).get('archived_files', 0) if isinstance(existing, dict) else 0
    if existing_count > fresh_archive['summary']['archived_files']:
        archive = existing
        print('loaded existing DtN archive to preserve deleted source snapshots')
for row in archive['files']:
    name = Path(row['path']).name
    m = re.match(r'act(\d+)_', name)
    row['act_group'] = f'act{m.group(1)}' if m else 'misc'
archive['summary']['by_act_group'] = {}
for row in archive['files']:
    archive['summary']['by_act_group'][row['act_group']] = archive['summary']['by_act_group'].get(row['act_group'], 0) + 1
write_report_json(archive, archive_path)
print(json.dumps(archive['summary'], ensure_ascii=False, indent=2))
print(f'\nwrote {archive_path.relative_to(NOTEBOOK_DIR)}')


loaded existing DtN archive to preserve deleted source snapshots
{
  "archived_files": 122,
  "archived_lines": 18927,
  "archived_bytes": 1050293,
  "by_lane": {
    "open_boundary_src_api_or_validation": 119,
    "validation_test_candidate": 2,
    "validation_test_or_src_api_locked": 1
  },
  "by_group": {
    "DtN_spectrum": 122
  },
  "by_act_group": {
    "act0": 6,
    "act1": 7,
    "act2": 15,
    "act3": 7,
    "act4": 7,
    "act5": 9,
    "act6": 13,
    "act7": 40,
    "act8": 17,
    "misc": 1
  }
}

wrote kelvin_dtn_spectrum_archive_results.json


In [2]:
from IPython.display import Markdown, display

display(Markdown('## DtN scripts by act group'))
display(Markdown(markdown_table([
    {'act_group': k, 'files': v} for k, v in sorted(archive['summary']['by_act_group'].items())
], ['act_group', 'files'], max_rows=20)))

display(Markdown('## Referenced / locked DtN scripts'))
locked = [row for row in archive['files'] if row['reference_hit_count_reported'] or row['validation_named']]
display(Markdown(markdown_table([
    {
        'path': row['path'],
        'lane': row['migration_lane'],
        'refs': row['reference_hit_count_reported'],
        'validation': row['validation_named'],
        'sha256': row['source']['sha256'][:12],
    }
    for row in locked
], ['path', 'lane', 'refs', 'validation', 'sha256'], max_rows=80)))
print(f'locked scripts shown: {len(locked)}')

## DtN scripts by act group

| act_group | files |
| --- | --- |
| act0 | 6 |
| act1 | 7 |
| act2 | 15 |
| act3 | 7 |
| act4 | 7 |
| act5 | 9 |
| act6 | 13 |
| act7 | 40 |
| act8 | 17 |
| misc | 1 |

## Referenced / locked DtN scripts

| path | lane | refs | validation | sha256 |
| --- | --- | --- | --- | --- |
| examples/kelvin_transformation/DtN_spectrum/act2_15_nonsymmetric_validation.py | validation_test_candidate | 0 | True | 28aaa9c68a8a |
| examples/kelvin_transformation/DtN_spectrum/act6_02_cln_dtn_cauer.py | validation_test_or_src_api_locked | 1 | False | 4c58c49ca33e |
| examples/kelvin_transformation/DtN_spectrum/act7_32_ie_3d_assembly_validation.py | validation_test_candidate | 0 | True | 8fa368a4ab21 |

locked scripts shown: 3


## Migration rule

The `act6_*` and `act7_*` families already have production or validation descendants. When pruning, update those descendants' comments/docs first. The `act8_*` stream-function-with-iron family should be coordinated with the stream-function panel/API migration rather than deleted as a Kelvin-only cleanup.